# TF32 — Eval ViSoBERT trên Google Colab

Notebook chạy `nd-khoa/vihsd-uit-visobert-v2` trên 30 câu rồi 3.000 câu. Sau đó notebook nhận cache OpenAI từ máy của bạn và so sánh OpenAI, ViSoBERT cùng phương án kết hợp.

Trước khi chạy, chọn **Runtime → Change runtime type → T4 GPU**. Chạy từng cell từ trên xuống. Nếu Colab không cấp GPU, notebook vẫn chạy bằng CPU.

In [ ]:
%pip install -q "transformers==4.51.0" "tokenizers==0.21.2" sentencepiece safetensors \
    "pydantic>=2.6,<3" pyyaml

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

REPO_DIR = Path("/content/tripflix-ai")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "develop",
            "https://github.com/Sukemcute/tripflix-ai.git",
            str(REPO_DIR),
        ],
        check=True,
    )

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
from app.services.summary.pii import mask_pii  # noqa: E402 - repo path is set above

DATA_30 = REPO_DIR / "evals/moderation/vi_30.jsonl"
DATA_3K = REPO_DIR / "evals/moderation/vi_3k.jsonl"
OUTPUT_DIR = Path("/content/visobert-eval")
OUTPUT_DIR.mkdir(exist_ok=True)
assert DATA_30.exists() and DATA_3K.exists()
print(f"Repo: {REPO_DIR}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.cuda.is_available()}")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "nd-khoa/vihsd-uit-visobert-v2"
MODEL_REVISION = "5e8e892f45bf3d52bc3d4c780b6235b65714f07b"
LABELS = ("clean", "offensive", "hate")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64 if DEVICE.type == "cuda" else 16

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False
)
model, loading_info = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
    use_safetensors=True,
    output_loading_info=True,
)
assert not loading_info["missing_keys"], loading_info
assert not loading_info["mismatched_keys"], loading_info
assert model.config.num_labels == len(LABELS)
model.to(DEVICE).eval()
print(f"Model loaded on {DEVICE}; batch size {BATCH_SIZE}")

In [ ]:
def load_rows(dataset_path: Path) -> tuple[bytes, list[dict]]:
    raw = dataset_path.read_bytes()
    rows = [json.loads(line) for line in raw.decode("utf-8-sig").splitlines() if line.strip()]
    assert rows and len({row["id"] for row in rows}) == len(rows)
    return raw, rows


def run_visobert(dataset_path: Path, output_path: Path) -> tuple[list[dict], dict]:
    raw, rows = load_rows(dataset_path)
    predictions = {}
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    started_at = time.perf_counter()

    with torch.inference_mode():
        offsets = range(0, len(rows), BATCH_SIZE)
        for offset in tqdm(offsets, total=(len(rows) + BATCH_SIZE - 1) // BATCH_SIZE):
            batch = rows[offset : offset + BATCH_SIZE]
            encoded = tokenizer(
                [mask_pii(row["text"]) for row in batch],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            ).to(DEVICE)
            probabilities = model(**encoded).logits.softmax(dim=-1).cpu()
            assert torch.isfinite(probabilities).all()
            for row, values in zip(batch, probabilities.tolist(), strict=True):
                clean, offensive, hate = values
                predictions[row["id"]] = {
                    "clean": clean,
                    "offensive": offensive,
                    "hate": hate,
                    "toxic": offensive + hate,
                }

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - started_at
    run = {
        "model": MODEL_ID,
        "revision": MODEL_REVISION,
        "dataset": dataset_path.name,
        "sha256": hashlib.sha256(raw).hexdigest(),
        "device": str(DEVICE),
        "batch_size": BATCH_SIZE,
        "max_length": 256,
        "seconds": elapsed_seconds,
        "predictions": predictions,
    }
    output_path.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    print(
        f"{len(rows)} predictions in {elapsed_seconds:.2f}s "
        f"({len(rows) / elapsed_seconds:.1f} rows/s) → {output_path}"
    )
    return rows, run

## Chạy thử 30 câu

Cell này xác nhận model tải được, thứ tự nhãn hợp lý và file kết quả có đủ 30 ID.

In [ ]:
rows_30, run_30 = run_visobert(DATA_30, OUTPUT_DIR / "visobert-30.json")
assert len(run_30["predictions"]) == 30
preview = []
for row in rows_30:
    scores = run_30["predictions"][row["id"]]
    predicted_label = max(LABELS, key=lambda label: scores[label])
    preview.append(
        {
            "id": row["id"],
            "expected": "toxic" if row["toxic"] else "clean",
            "predicted": predicted_label,
            "toxic_score": scores["toxic"],
            "masked_text": mask_pii(row["text"]),
        }
    )
display(pd.DataFrame(preview).head(12))

## Chạy 3.000 câu

Giữ tab Colab mở cho đến khi progress bar hoàn tất. Kết quả được lưu tạm trong phiên Colab.

In [ ]:
rows_3k, run_3k = run_visobert(DATA_3K, OUTPUT_DIR / "visobert-3k.json")
assert len(run_3k["predictions"]) == 3000

## Upload cache OpenAI

Chọn file `D:\Sukem\VinUni\VSF\tripflix\tripflix-ai\evals\moderation\.eval_cache.json` từ máy của bạn. File chỉ được đưa vào phiên Colab hiện tại.

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Chỉ chọn một file .eval_cache.json"
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
OPENAI_CACHE_PATH = OUTPUT_DIR / "openai-eval-cache.json"
OPENAI_CACHE_PATH.write_bytes(uploaded_bytes)
openai_cache = json.loads(uploaded_bytes.decode("utf-8-sig"))
assert len(openai_cache) == 3000, f"Expected 3000 cache entries, got {len(openai_cache)}"
print(f"Loaded {len(openai_cache)} OpenAI cache entries from {uploaded_name}")

## So sánh kết quả

Ngưỡng `0.50` là điểm bắt đầu để khảo sát. Bảng hiển thị kết quả của OpenAI, ViSoBERT và phép OR giữa hai model trên đúng 3.000 câu.

In [ ]:
VISOBERT_THRESHOLD = 0.50

assert 0 <= VISOBERT_THRESHOLD <= 1
raw_3k, rows_3k = load_rows(DATA_3K)
assert run_3k["sha256"] == hashlib.sha256(raw_3k).hexdigest()
assert set(run_3k["predictions"]) == {row["id"] for row in rows_3k}

openai_flags = {}
visobert_flags = {}
for row in rows_3k:
    row_id = row["id"]
    cached = openai_cache[row_id]
    assert cached["text"] == row["text"], row_id
    assert cached["model"] == "omni-moderation-latest", row_id
    openai_flags[row_id] = bool(cached["flagged"])
    visobert_flags[row_id] = run_3k["predictions"][row_id]["toxic"] >= VISOBERT_THRESHOLD

all_flags = {
    "OpenAI": openai_flags,
    "ViSoBERT": visobert_flags,
    "OpenAI OR ViSoBERT": {
        row["id"]: openai_flags[row["id"]] or visobert_flags[row["id"]] for row in rows_3k
    },
}


def calculate_metrics(rows: list[dict], flags: dict[str, bool]) -> dict:
    tp = sum(row["toxic"] and flags[row["id"]] for row in rows)
    fp = sum(not row["toxic"] and flags[row["id"]] for row in rows)
    fn = sum(row["toxic"] and not flags[row["id"]] for row in rows)
    tn = sum(not row["toxic"] and not flags[row["id"]] for row in rows)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "total": len(rows),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "accuracy": (tp + tn) / len(rows),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fp / (fp + tn),
    }


summary_rows = []
baseline = calculate_metrics(rows_3k, openai_flags)
for model_name, flags in all_flags.items():
    metrics = calculate_metrics(rows_3k, flags)
    summary_rows.append(
        {
            "model": model_name,
            **metrics,
            "delta_tp_vs_openai": metrics["tp"] - baseline["tp"],
            "delta_fp_vs_openai": metrics["fp"] - baseline["fp"],
        }
    )
summary = pd.DataFrame(summary_rows)
display(
    summary.style.format(
        {
            "accuracy": "{:.2%}",
            "precision": "{:.2%}",
            "recall": "{:.2%}",
            "f1": "{:.2%}",
            "fpr": "{:.2%}",
        }
    )
)

by_kind_rows = []
for model_name, flags in all_flags.items():
    for kind in sorted({row["kind"] for row in rows_3k}):
        kind_rows = [row for row in rows_3k if row["kind"] == kind]
        correct = sum(flags[row["id"]] == row["toxic"] for row in kind_rows)
        by_kind_rows.append(
            {
                "model": model_name,
                "kind": kind,
                "expected": "toxic" if kind_rows[0]["toxic"] else "clean",
                "total": len(kind_rows),
                "correct": correct,
                "correct_rate": correct / len(kind_rows),
                "flagged": sum(flags[row["id"]] for row in kind_rows),
            }
        )
by_kind = pd.DataFrame(by_kind_rows)
display(by_kind.style.format({"correct_rate": "{:.2%}"}))

In [ ]:
import zipfile

disagreement_rows = []
for row in rows_3k:
    row_id = row["id"]
    if openai_flags[row_id] != visobert_flags[row_id]:
        scores = run_3k["predictions"][row_id]
        disagreement_rows.append(
            {
                "id": row_id,
                "kind": row["kind"],
                "expected_toxic": row["toxic"],
                "openai_flagged": openai_flags[row_id],
                "visobert_flagged": visobert_flags[row_id],
                "visobert_toxic_score": scores["toxic"],
                "visobert_clean": scores["clean"],
                "visobert_offensive": scores["offensive"],
                "visobert_hate": scores["hate"],
                "masked_text": mask_pii(row["text"]),
            }
        )
disagreements = pd.DataFrame(disagreement_rows)
summary_path = OUTPUT_DIR / "comparison-summary.csv"
by_kind_path = OUTPUT_DIR / "comparison-by-kind.csv"
disagreements_path = OUTPUT_DIR / "disagreements.csv"
summary.to_csv(summary_path, index=False)
by_kind.to_csv(by_kind_path, index=False)
disagreements.to_csv(disagreements_path, index=False)

archive_path = Path("/content/tf32-visobert-eval-results.zip")
files_to_export = [
    OUTPUT_DIR / "visobert-30.json",
    OUTPUT_DIR / "visobert-3k.json",
    summary_path,
    by_kind_path,
    disagreements_path,
]
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in files_to_export:
        archive.write(file_path, arcname=file_path.name)
print(f"Exported {archive_path}; {len(disagreements)} model disagreements")
files.download(str(archive_path))